# Chapter 5 -- Harness Anatomy (Practice)

Work through this notebook **after reading** `notes/ch05-harness-anatomy.md`. This chapter refactors Chapter 2's monolithic loop into a `Harness` class with pluggable components and lifecycle hooks (notes Section 7) -- the same `PreToolUse`/`PostCompact`-style events the Claude Agent SDK exposes, built here from scratch so you can see exactly what they do.

You'll implement two hooks matching the two concrete examples from the notes: a `PreToolUse` hook that makes writing outside the workspace **structurally impossible** (not just discouraged), and a `PostCompact` hook that persists a decision log. Everything is verified offline -- no API key needed for either exercise.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import anthropic


def _find_and_load_env() -> None:
    """Walk up from the current working directory to find and load a repo-root .env file, if one exists."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    print("No .env file found -- copy .env.example to .env at the repo root to enable the real-model sections.")


_find_and_load_env()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_NAME = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-sonnet-5")


def test_connection(client, model_name):
    """Send a trivial ping to confirm the Bedrock connection actually works."""
    print(f"Testing connection to Bedrock (model={model_name})...")
    try:
        response = client.messages.create(
            model=model_name, max_tokens=10,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = next((b.text for b in response.content if b.type == "text"), "")
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("The rest of this notebook still works fully offline -- this cell")
        print("only matters for the optional real-model section at the end.")


if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env -- skipping connection test.")
    print("The rest of this notebook still works fully offline.")
else:
    client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    test_connection(client, MODEL_NAME)


## The Workspace and Its Tools

`write_file` and `read_file` below are deliberately **naive** -- `write_file` has no path-safety check of its own. That's the point: notes Section 7 makes a specific claim, that a hook can make a failure "structurally impossible to repeat" without the tool itself needing to know anything about safety. If `write_file` had its own check too, you'd never be able to tell whether the hook or the tool caught an unsafe write. This way, the hook you build in Exercise 1 is the *only* thing standing between the model and a file outside the workspace.

In [ ]:
WORKSPACE_DIR = Path.cwd() / "ch05_workspace"
WORKSPACE_DIR.mkdir(exist_ok=True)


def write_file(path: str, content: str) -> str:
    """Write `content` to `path`, resolved relative to WORKSPACE_DIR. No safety check -- see the note above."""
    target = WORKSPACE_DIR / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"wrote {len(content)} chars to {path}"


def read_file(path: str) -> str:
    """Read the contents of `path`, resolved relative to WORKSPACE_DIR."""
    target = WORKSPACE_DIR / path
    if not target.is_file():
        raise FileNotFoundError(f"{path} not found in {WORKSPACE_DIR.name}/")
    return target.read_text()


TOOL_DISPATCH = {"write_file": write_file, "read_file": read_file}

TOOL_SCHEMAS = [
    {
        "name": "write_file",
        "description": "Write text content to a file inside the workspace.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "File path relative to the workspace, e.g. 'report.txt'"},
                "content": {"type": "string", "description": "Text content to write"},
            },
            "required": ["path", "content"],
        },
    },
    {
        "name": "read_file",
        "description": "Read the full contents of a file inside the workspace.",
        "input_schema": {
            "type": "object",
            "properties": {"path": {"type": "string", "description": "File path relative to the workspace"}},
            "required": ["path"],
        },
    },
]

print(f"Workspace ready at {WORKSPACE_DIR}")
print("Tools registered:", [t["name"] for t in TOOL_SCHEMAS])


## The `Harness` Class

`Harness` wraps Chapter 2's loop mechanics (message list, tool dispatch, `stop_reason` branching) with two additions notes Section 2 named as separate components: a **hook registry** (Section 7) and a **startup banner** (Section 8's "the harness is a versioned, inspectable artifact" made literal -- you can print exactly what's active). Hooks are just plain functions registered against an event name; `Harness` calls every registered hook at the right moment and obeys what a `pre_tool_use` hook returns, exactly like the real SDK's `PreToolUse` mechanic from the notes.

In [ ]:
class Harness:
    """
    Wraps a tool-calling agent loop with a pluggable hook registry.
    Supported events: pre_tool_use, post_tool_use, pre_compact, post_compact.
    """

    def __init__(self, model, tool_schemas, tool_dispatch, workspace_dir, version="v1.0"):
        self.model = model
        self.tool_schemas = tool_schemas
        self.tool_dispatch = tool_dispatch
        self.workspace_dir = workspace_dir
        self.version = version
        self.hooks = {"pre_tool_use": [], "post_tool_use": [], "pre_compact": [], "post_compact": []}

    def register_hook(self, event, callback):
        if event not in self.hooks:
            raise ValueError(f"unknown hook event '{event}'")
        self.hooks[event].append(callback)

    def print_startup_banner(self):
        print("-" * 60)
        print(f"HARNESS STARTUP -- version {self.version}")
        print("-" * 60)
        print(f"  model:      {self.model}")
        print(f"  workspace:  {self.workspace_dir}")
        print(f"  tools:      {[t['name'] for t in self.tool_schemas]}")
        for event, callbacks in self.hooks.items():
            names = [cb.__name__ for cb in callbacks] if callbacks else ["(none registered)"]
            print(f"  hooks[{event}]: {names}")
        print("-" * 60)

    def execute_tool_call(self, block):
        """Run one tool_use block through the pre/post_tool_use hook chain. Returns (content_str, is_error)."""
        for hook in self.hooks["pre_tool_use"]:
            decision = hook(block.name, block.input, self.workspace_dir)
            if decision and decision.get("permissionDecision") == "deny":
                content_str, is_error = f"Error: {decision['permissionDecisionReason']}", True
                for post_hook in self.hooks["post_tool_use"]:
                    post_hook(block.name, block.input, content_str, is_error)
                return content_str, is_error

        fn = self.tool_dispatch.get(block.name)
        try:
            if fn is None:
                raise ValueError(f"no such tool '{block.name}'")
            content_str, is_error = str(fn(**block.input)), False
        except Exception as exc:
            content_str, is_error = f"Error: {exc}", True

        for post_hook in self.hooks["post_tool_use"]:
            post_hook(block.name, block.input, content_str, is_error)
        return content_str, is_error

    def compact(self, summary_text):
        """Simulate Chapter 4's compaction event, firing pre/post_compact hooks around it."""
        for hook in self.hooks["pre_compact"]:
            hook(summary_text)
        for hook in self.hooks["post_compact"]:
            hook(summary_text)
        return summary_text

    def run_agent_loop(self, client, messages, max_steps=10):
        """Chapter 2's loop, unchanged in spirit, routed through execute_tool_call."""
        for step in range(1, max_steps + 1):
            response = client.messages.create(model=self.model, max_tokens=1024, tools=self.tool_schemas, messages=messages)
            print(f"STEP {step} | stop_reason={response.stop_reason}")

            if response.stop_reason == "end_turn":
                final_text = next((b.text for b in response.content if b.type == "text"), "")
                print(f"  Final answer: {final_text}")
                return final_text, step

            messages.append({"role": "assistant", "content": response.content})
            results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue
                content_str, is_error = self.execute_tool_call(block)
                status = "ERROR" if is_error else "ok"
                print(f"    tool_use {block.name}({block.input}) -> [{status}] {content_str[:70]}")
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": content_str, "is_error": is_error})
            messages.append({"role": "user", "content": results})

        print(f"Hit max_steps ({max_steps}) without a natural stop.")
        return None, max_steps


print("Harness class defined.")


## Testing Offline: the Scripted Fake Client

Same approach as Chapter 2: a `FakeClient` that mimics `.messages.create()`'s exact response shape, so both exercises below are verified deterministically with no API key. `FAKE_SCRIPT` scripts a model that first attempts an unsafe write (outside the workspace), then a safe one, then reads it back to confirm.

In [ ]:
from types import SimpleNamespace


def _text(text):
    return SimpleNamespace(type="text", text=text)


def _tool_use(tool_id, name, tool_input):
    return SimpleNamespace(type="tool_use", id=tool_id, name=name, input=tool_input)


def _response(stop_reason, content):
    return SimpleNamespace(stop_reason=stop_reason, content=content, usage=SimpleNamespace(input_tokens=100, output_tokens=20))


FAKE_SCRIPT = [
    _response("tool_use", [_tool_use("call_1", "write_file", {"path": "../escape.txt", "content": "malicious payload"})]),
    _response("tool_use", [_tool_use("call_2", "write_file", {"path": "report.txt", "content": "Q3 summary: revenue up 12%."})]),
    _response("tool_use", [_tool_use("call_3", "read_file", {"path": "report.txt"})]),
    _response("end_turn", [_text("Wrote report.txt after one blocked attempt outside the workspace; confirmed its contents.")]),
]


class FakeClient:
    def __init__(self, script):
        self.script = list(script)
        self.messages = SimpleNamespace(create=self._create)

    def _create(self, **kwargs):
        if not self.script:
            raise RuntimeError("FakeClient script exhausted -- the loop asked for more steps than scripted")
        return self.script.pop(0)


print(f"FakeClient ready with a {len(FAKE_SCRIPT)}-step scripted trajectory.")


## Exercise 1 -- A `PreToolUse` Hook That Makes Escaping the Workspace Impossible

Implement `block_writes_outside_workspace`, a `pre_tool_use` hook (notes Section 7's `.env`-blocking example, generalized to any path outside the workspace). It only cares about `write_file` calls; for those, resolve the target path against `workspace_dir` and deny it if it resolves outside -- covering both `../` traversal and an absolute path that escapes the workspace entirely.

In [ ]:
def block_writes_outside_workspace(tool_name, tool_input, workspace_dir):
    """
    PreToolUse-style hook. Only inspects write_file calls.

    Returns:
      None                                     -- allow (not a write_file call, or the path is safe)
      {"permissionDecision": "deny",
       "permissionDecisionReason": <str>}       -- block the call
    """
    if tool_name != "write_file":
        return None
    # TODO: resolve tool_input["path"] relative to workspace_dir (Path
    # joining an absolute path replaces the base entirely -- that's exactly
    # the escape case you need to catch). Use Path.relative_to() against
    # workspace_dir.resolve() -- it raises ValueError when the path is NOT
    # inside the workspace. Return a deny decision in that case, else None.
    raise NotImplementedError("TODO: implement block_writes_outside_workspace")


In [ ]:
safe = block_writes_outside_workspace("write_file", {"path": "report.txt"}, WORKSPACE_DIR)
assert safe is None, "a path inside the workspace should be allowed (None)"

nested_safe = block_writes_outside_workspace("write_file", {"path": "sub/dir/file.txt"}, WORKSPACE_DIR)
assert nested_safe is None, "a nested path still inside the workspace should be allowed"

dotdot = block_writes_outside_workspace("write_file", {"path": "../escape.txt"}, WORKSPACE_DIR)
assert dotdot is not None and dotdot["permissionDecision"] == "deny", "'../escape.txt' must be denied"
assert "escape.txt" in dotdot["permissionDecisionReason"], "the reason should name the offending path"

absolute = block_writes_outside_workspace("write_file", {"path": "/etc/passwd"}, WORKSPACE_DIR)
assert absolute is not None and absolute["permissionDecision"] == "deny", "an absolute path escaping the workspace must be denied"

ignored = block_writes_outside_workspace("read_file", {"path": "../escape.txt"}, WORKSPACE_DIR)
assert ignored is None, "the hook should only inspect write_file calls -- read_file is not its concern"

print("Exercise 1 PASSED -- safe paths (including nested ones) are allowed,")
print("both '../' traversal and absolute-path escapes are denied with a clear reason,")
print("and non-write tools are left alone.")


## Exercise 2 -- A `PostCompact` Hook That Persists a Decision Log

Implement `make_decision_log_hook`, a factory that returns a `post_compact` hook appending each compaction summary to a log file -- notes Section 6's "decision log" pattern, wired to the exact lifecycle moment (`PostCompact`) notes Section 7 named for it. Each call should **append** one line, never overwrite -- the whole point of a log is that history accumulates.

In [ ]:
def make_decision_log_hook(log_path):
    """
    Returns a post_compact hook: hook(summary_text) -> None, which appends
    summary_text as one line to log_path, creating the file if needed.
    """
    def hook(summary_text):
        # TODO: open log_path in append mode ("a") and write summary_text
        # followed by a newline. Path.open() creates the file if it
        # doesn't exist yet -- no need to check existence first.
        raise NotImplementedError("TODO: implement make_decision_log_hook's inner hook")
    return hook


In [ ]:
TEST_LOG_PATH = Path.cwd() / "ch05_test_decisions.log"
if TEST_LOG_PATH.exists():
    TEST_LOG_PATH.unlink()

test_hook = make_decision_log_hook(TEST_LOG_PATH)
test_hook("Summary A: chose Postgres over SQLite for concurrent writes.")
test_hook("Summary B: switched retry policy to error-informed retry.")

assert TEST_LOG_PATH.exists(), "the log file should be created on first write"
lines = TEST_LOG_PATH.read_text().splitlines()
assert len(lines) == 2, f"expected 2 log lines after 2 calls, got {len(lines)}"
assert "Summary A" in lines[0] and "Summary B" in lines[1], "each call should append, not overwrite, the previous entry"

TEST_LOG_PATH.unlink()  # clean up the throwaway test log
print("Exercise 2 PASSED -- the decision log hook appends one line per call")
print("and creates the file on first use.")


## Assembling the Harness and Running the Integrated Demo

Both hooks now get registered on a real `Harness` instance -- this is the startup banner (notes Section 8) actually enumerating what's active. Then `FAKE_SCRIPT`'s trajectory runs through `harness.run_agent_loop`: a blocked write, a successful write, and a read-back confirming it -- the same demonstration Section 7 walked through by hand, now actually executing.

In [ ]:
DECISION_LOG_PATH = Path.cwd() / "ch05_decisions.log"
if DECISION_LOG_PATH.exists():
    DECISION_LOG_PATH.unlink()

harness = Harness(model=MODEL_NAME, tool_schemas=TOOL_SCHEMAS, tool_dispatch=TOOL_DISPATCH, workspace_dir=WORKSPACE_DIR)
harness.register_hook("pre_tool_use", block_writes_outside_workspace)
harness.register_hook("post_compact", make_decision_log_hook(DECISION_LOG_PATH))
harness.print_startup_banner()

print()
print("-" * 60)
print("RUNNING SCRIPTED TRAJECTORY")
print("-" * 60)
fake_client = FakeClient(FAKE_SCRIPT)
fake_messages = [{"role": "user", "content": "Write a Q3 report to the workspace, then read it back to confirm."}]
final_text, steps_taken = harness.run_agent_loop(fake_client, fake_messages)

print()
escape_path = WORKSPACE_DIR.parent / "escape.txt"
assert not escape_path.exists(), "the blocked write must never have touched the filesystem"
report_path = WORKSPACE_DIR / "report.txt"
assert report_path.is_file(), "the safe write should have succeeded"
assert "revenue up 12%" in report_path.read_text(), "report.txt should contain the content the model wrote"
print("Confirmed: the escape attempt never touched disk, and the safe write is present with correct content.")


## Triggering Compaction and Inspecting the Decision Log

`harness.compact()` simulates Chapter 4's compaction event, firing `pre_compact` then `post_compact` hooks. Called twice with two different summaries, the log file should end up with exactly two lines -- proof the hook is wired to the harness, not just tested in isolation.

In [ ]:
harness.compact("Compaction 1: wrote Q3 report after blocking one unsafe write attempt outside the workspace.")
harness.compact("Compaction 2: confirmed report.txt contents via read_file; task complete.")

print(f"Decision log at {DECISION_LOG_PATH}:")
print("-" * 60)
log_lines = DECISION_LOG_PATH.read_text().splitlines()
for line in log_lines:
    print(f"  {line}")
print("-" * 60)

assert len(log_lines) == 2, f"expected 2 decision-log entries after 2 compact() calls, got {len(log_lines)}"
print("Confirmed: two compact() calls produced exactly two persisted decision-log entries.")


## Notes Section 11, Made Concrete: Why One Change Per Version Is the Only Attributable Design

A short, direct illustration of the notes' dry-run: five harness versions, first changing two components per version (ambiguous), then changing exactly one (attributable). Nothing here is a TODO -- it's worth seeing the two tables actually printed side by side.

In [ ]:
two_at_a_time = [
    {"version": "v2", "changed": ["permissions", "context_delivery"], "score": 68},
    {"version": "v3", "changed": ["context_delivery", "memory"], "score": 65},
    {"version": "v4", "changed": ["memory", "tool_interface"], "score": 71},
    {"version": "v5", "changed": ["tool_interface", "permissions"], "score": 69},
]
one_at_a_time = [
    {"version": "v2", "changed": ["permissions"], "score": 68},
    {"version": "v3", "changed": ["context_delivery"], "score": 65},
    {"version": "v4", "changed": ["memory"], "score": 71},
    {"version": "v5", "changed": ["tool_interface"], "score": 69},
]
BASELINE_SCORE = 62


def print_attribution_table(title, versions, attributable):
    print("-" * 60)
    print(title)
    print("-" * 60)
    prev_score = BASELINE_SCORE
    print(f"  v1 (baseline): score={BASELINE_SCORE}")
    for v in versions:
        delta = v["score"] - prev_score
        sign = "+" if delta >= 0 else ""
        verdict = f"attributable to: {v['changed'][0]}" if attributable else "AMBIGUOUS -- cannot tell which change caused this"
        print(f"  {v['version']}: changed={v['changed']}  score={v['score']}  delta={sign}{delta}  ({verdict})")
        prev_score = v["score"]
    print()


print_attribution_table("TWO COMPONENTS CHANGED PER VERSION", two_at_a_time, attributable=False)
print_attribution_table("ONE COMPONENT CHANGED PER VERSION", one_at_a_time, attributable=True)
print("Same scores, same deltas -- the only difference is whether each version's")
print("delta has exactly one possible explanation. That difference is entirely")
print("in the experiment design, not in any analysis performed after the fact.")


## Optional -- Run the Same Harness Against a Real Claude Model

The exact same `Harness`, with both hooks still registered, run against a live model via `AnthropicBedrockMantle` (Claude Sonnet, through AWS Bedrock). If the real model attempts anything resembling an unsafe write, the same `block_writes_outside_workspace` hook is what would stop it -- this is a genuinely live run, not scripted.

In [ ]:
RUN_REAL_HARNESS_DEMO = False

REAL_DEMO_PROMPT = (
    "Write a short Q3 summary to a file in the workspace, then read it back "
    "to confirm its contents. Use the tools -- do not guess."
)


def run_real_harness_demo():
    if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
        print("Skipping real harness demo: AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env.")
        return

    real_client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    real_messages = [{"role": "user", "content": REAL_DEMO_PROMPT}]
    try:
        final_text, steps_taken = harness.run_agent_loop(real_client, real_messages)
    except Exception as exc:
        print(f"Real harness run failed: {type(exc).__name__}: {exc}")
        return

    print()
    print(f"Real run finished in {steps_taken} step(s).")
    print(f"Final answer: {final_text}")


if RUN_REAL_HARNESS_DEMO:
    run_real_harness_demo()
else:
    print("RUN_REAL_HARNESS_DEMO is False -- running in offline/scripted mode only.")
    print("Flip it to True to run this exact harness (hooks included) against a")
    print("real Claude model via Bedrock.")


## Key Takeaways

You've built a real `Harness` class -- not a toy -- with a hook registry, a startup banner that enumerates exactly what's active, and two working hooks matching notes Section 7's own examples. The escape-attempt test in this notebook is worth remembering on its own: `write_file` has no safety logic of its own, and yet an unsafe write was structurally impossible to complete, because the hook never had to trust the model's intentions -- it only ever looked at a tool name and a resolved path. That's notes Section 1's whole claim about what a harness is for, made literal in code you just ran.

**Connection forward:** Chapter 6 opens up the one component this chapter named but left as a placeholder -- **loop control** -- and asks how an agent decides it's actually *done*: a verifier ladder, defeating victory-declaration bias, retry policy, and stop rules that check more than just `stop_reason == "end_turn"`.